## Tahap 1 – Penyimpanan Terdistribusi dan Ingesti (HDFS)

### Tujuan
1. Membangun direktori terdistribusi di HDFS (`/tidarmart/raw`, `/tidarmart/stream_input`, `/tidarmart/curated`, dan `/tidarmart/models`) untuk memisahkan  data mentah, data aliran, data bersih, dan wadah untuk model machine learning.
2. Mengunggah seluruh dataset multi-format (CSV transaksi batch, JSON Lines pelanggan dan produk, dan CSV micro-batch streaming) ke HDFS agar bisa digunakan sebagai sumber data pada tahap selanjutnya. seluruh tahap pemrosesan selanjutnya membaca data langsung dari `hdfs://localhost:9000/tidarmart/...` dan bukan dari path lokal.
3. Memverifikasi keberhasilan upload data serta mendokumentasikan bukti karakteristik **Volume** melalui pemeriksaan ukuran berkas (`hdfs dfs -du -h`), jumlah blok, status kesehatan, dan faktor replikasi (`hdfs fsck`).

In [2]:
%%bash
#membuat direktori yang diperluakan di HDFS
hdfs dfs -mkdir -p /tidarmart/raw/transaksi
hdfs dfs -mkdir -p /tidarmart/raw/pelanggan
hdfs dfs -mkdir -p /tidarmart/raw/produk
hdfs dfs -mkdir -p /tidarmart/stream_input
hdfs dfs -mkdir -p /tidarmart/curated
hdfs dfs -mkdir -p /tidarmart/models

# 2. Unggah data ke HDFS (-f agar menimpa jika dijalankan ulang)
hdfs dfs -put -f data_proyek/transaksi/*.csv /tidarmart/raw/transaksi/
hdfs dfs -put -f data_proyek/pelanggan.json /tidarmart/raw/pelanggan/
hdfs dfs -put -f data_proyek/produk.json /tidarmart/raw/produk/
hdfs dfs -put -f data_proyek/stream_input/*.csv /tidarmart/stream_input/

# 3. Bukti ukuran, jumlah blok, dan faktor replikasi
hdfs dfs -du -h /tidarmart/raw
hdfs dfs -du -h /tidarmart/stream_input
hdfs fsck /tidarmart -files -blocks

6.0 M    6.0 M    /tidarmart/raw/pelanggan
684.3 K  684.3 K  /tidarmart/raw/produk
187.2 M  187.2 M  /tidarmart/raw/transaksi
479.9 K  479.9 K  /tidarmart/stream_input/stream_01.csv
479.5 K  479.5 K  /tidarmart/stream_input/stream_02.csv
479.5 K  479.5 K  /tidarmart/stream_input/stream_03.csv
479.2 K  479.2 K  /tidarmart/stream_input/stream_04.csv
479.7 K  479.7 K  /tidarmart/stream_input/stream_05.csv
479.0 K  479.0 K  /tidarmart/stream_input/stream_06.csv
479.5 K  479.5 K  /tidarmart/stream_input/stream_07.csv
479.1 K  479.1 K  /tidarmart/stream_input/stream_08.csv
479.3 K  479.3 K  /tidarmart/stream_input/stream_09.csv
479.5 K  479.5 K  /tidarmart/stream_input/stream_10.csv
479 K    479 K    /tidarmart/stream_input/stream_11.csv
479.5 K  479.5 K  /tidarmart/stream_input/stream_12.csv
479.6 K  479.6 K  /tidarmart/stream_input/stream_13.csv
479.5 K  479.5 K  /tidarmart/stream_input/stream_14.csv
479.8 K  479.8 K  /tidarmart/stream_input/stream_15.csv
479.0 K  479.0 K  /tidarmart/strea

Connecting to namenode via http://localhost:9870/fsck?ugi=cahyoalim&files=1&blocks=1&path=%2Ftidarmart


FSCK started by cahyoalim (auth:SIMPLE) from /127.0.0.1 for path /tidarmart at Sun Nov 22 00:04:12 WIB 2026

/tidarmart <dir>
/tidarmart/curated <dir>
/tidarmart/models <dir>
/tidarmart/raw <dir>
/tidarmart/raw/pelanggan <dir>
/tidarmart/raw/pelanggan/pelanggan.json 6307387 bytes, replicated: replication=1, 1 block(s):  OK
0. BP-1724578924-127.0.1.1-1788753323439:blk_1073741893_1075 len=6307387 Live_repl=1

/tidarmart/raw/produk <dir>
/tidarmart/raw/produk/produk.json 700710 bytes, replicated: replication=1, 1 block(s):  OK
0. BP-1724578924-127.0.1.1-1788753323439:blk_1073741894_1076 len=700710 Live_repl=1

/tidarmart/raw/transaksi <dir>
/tidarmart/raw/transaksi/transaksi_part1.csv 49074339 bytes, replicated: replication=1, 1 block(s):  OK
0. BP-1724578924-127.0.1.1-1788753323439:blk_1073741889_1071 len=49074339 Live_repl=1

/tidarmart/raw/transaksi/transaksi_part2.csv 49077730 bytes, replicated: replication=1, 1 block(s):  OK
0. BP-1724578924-127.0.1.1-1788753323439:blk_1073741890_107

### Interpretasi Tahap 1

**1. Ringkasan Ingesti & Bukti Karakteristik Volume**
* Seluruh dataset berhasil diunggah ke HDFS dengan status **`HEALTHY`** (`Missing blocks: 0`, `Corrupt blocks: 0`). Total terdapat **26 berkas** yang tersimpan dalam **26 blok** tervalidasi dengan total ukuran **213.128.301 bytes (±203,3 MiB)**.
* Rincian kapasitas penyimpanan pada zona `raw` dan `stream_input`:
  * **Transaksi batch (`/tidarmart/raw/transaksi`):** **187,2 MB** (4 berkas CSV, masing-masing ~49 MB, total 2.010.000 baris) — menjadi acuan utama pembanding ukuran terhadap format Parquet pada Tahap 3.
  * **Pelanggan (`/tidarmart/raw/pelanggan`):** **6,0 MB** (50.000 dokumen JSON Lines).
  * **Produk (`/tidarmart/raw/produk`):** **684,3 KB** (5.000 dokumen JSON Lines).
  * **Micro-batch streaming (`/tidarmart/stream_input`):** 20 berkas CSV, masing-masing berukuran **±479–480 KB**.
* Setiap berkas menempati tepat **1 blok** karena ukuran berkas terbesar (~49 MB) masih berada di bawah batas ukuran blok bawaan HDFS (128 MB).

**2. Alasan Pemisahan Direktori `raw` dan `curated`**
* Direktori **`raw`** berfungsi sebagai zona penyimpanan data mentah yang bersifat *immutable* (*single source of truth*). Data tidak pernah diubah langsung di folder ini agar jika terjadi kesalahan logika pembersihan atau perubahan kebutuhan bisnis di masa depan, pipeline ETL tetap dapat dijalankan ulang dari data asli tanpa kehilangan informasi.
* Direktori **`curated`** khusus menyimpan data hasil pembersihan, validasi, dan transformasi dalam format kolumnar (Parquet) yang siap dikonsumsi secara cepat dan konsisten oleh tim analitik maupun model *machine learning*.

**3. Makna Faktor Replikasi = 1 pada Lingkungan *Pseudo-Distributed* vs. Klaster Nyata**
* Hasil `hdfs fsck` menunjukkan `Number of data-nodes: 1` dan `Default replication factor: 1` (`Average block replication: 1.0`), sehingga kolom ukuran asli dan ukuran terpakai pada `hdfs dfs -du -h` bernilai identik (misalnya `187.2 M` dan `187.2 M`).
* Pada lingkungan lokal (*pseudo-distributed*), seluruh layanan Hadoop berjalan pada satu mesin saja sehingga replikasi 1 sudah memadai untuk simulasi dan menghemat ruang penyimpanan, namun **tidak memiliki toleransi kesalahan (*fault tolerance*)**—jika penyimpanan lokal rusak, data tidak memiliki salinan cadangan.
* Sebaliknya, pada klaster produksi nyata yang terdiri atas banyak server, faktor replikasi umumnya ditetapkan sebesar **3** dan disebar ke beberapa *DataNode* serta rak berbeda. Skema ini menjamin ketersediaan tinggi (*high availability*) bila ada node yang mati sekaligus mendukung pembacaan paralel berbasis *data locality*.